# 世界モデルの全体像

世界モデルは、観測をそのまま眺めるだけでなく、内部状態を持ち、その状態を行動で進め、必要なら観測へ戻し、候補行動を比べるための考え方です。生成モデル、状態空間モデル、制御、系列モデルが同時に出てきますが、中心は「先を試して次の行動を選ぶ」ことにあります。


## 世界モデルは『きれいに作る』より『先を試す』ための道具です

生成モデルが強いのは、もっともらしい観測を作ることです。一方で世界モデルが本当に欲しいのは、行動を変えたら未来がどう変わるかを、実環境へ出る前に試せることです。

そのため世界モデルでは、観測をただ再現するだけでは、行動選択まで支えられません。見えていない内部状態を仮定し、その状態が行動でどう変わるかを持ち、必要なら観測へ戻し、さらにその予測を計画へ使える必要があります。読むときは『何をきれいに再現したいのか』より、『その予測で次の行動を考えられるか』を基準にすると迷いにくくなります。

## 前提

- 深層生成モデル分野の基本: 潜在変数、VAE、拡散の役割差
- 強化学習分野の基本: 状態、行動、報酬、ロールアウトの感覚
- ディープラーニング分野の基本: 系列や表現学習の見方

ただし、これらを完璧に暗記している必要はありません。分からない語が出ても、その場で『いま見えている量の話か、裏の状態の話か』『未来を当てたいのか、観測を戻したいのか』だけ追えれば読み進められます。必要なのは、『見えているもの』と『その裏の状態』を分けて考える姿勢です。単語の意味を全部覚えることより、いま何を予測していて、その予測を何に使いたいのかを追う方が大切です。

## 読み順

1. 世界モデルと生成モデルの関係: 何が共通で何が違うか
2. 状態空間モデル: 状態遷移と観測生成を分ける
3. 状態表現学習: 圧縮した状態が下流で使えるかを見る
4. 観測予測モデル: 自己回帰・マスク予測・拡散の違いを整理する
5. VAE と拡散: 圧縮主体か高品質復元主体かを比べる
6. 状態予測モデル: 1 ステップと長期ロールアウトを分けて見る
7. 制御モデルと MBRL: 行動を入れた計画評価へ進む
8. シミュレーションと CG: 動きと見え方を切り分ける
9. マルチモーダル世界モデル: 複数入力をどう統合するか
10. SSM と Transformer: 長い系列の持ち方を比べる

前半は『未来を考えるための骨格』、後半は『その予測をどう使うか』を読む構成です。迷ったら、まず 1 から 4 で骨組みを掴み、そのあと 5 以降で用途の違いを読む順番でも問題ありません。もし途中で難しく感じたら、いったん 2 の状態空間モデルと 4 の観測予測モデルへ戻ると、他のテーマの位置づけを立て直しやすくなります。

## 先に揃えたい語彙

- 状態 `state`: 見えているデータの裏で連続的に動いている内部表現
- 観測 `observation`: センサーや画像として実際に見えている量
- 遷移 `transition`: いまの状態と行動から次の状態を作る規則
- ロールアウト `rollout`: 予測した次状態を使ってさらに先まで進めること
- 計画 `planning`: 候補行動列を比べて次の行動を選ぶこと

この言葉の区別が曖昧だと、見た目の復元が上手いことと、未来予測や計画に使えることを同じだと思いやすくなります。逆に、この 5 語が分かれて見えるだけで、各テーマの狙いはかなり追いやすくなります。読んでいて迷ったら、まず『いま見ているのは state / observation / transition / rollout / planning のどれか』を言い直してみるのが有効です。

In [ ]:
z = [0.2, -0.1]
a = [1.0]
print('state z =', z)
print('action a =', a)
print('idea: use (z, a) to predict next state')

この最小例で見たいのは、観測画像やセンサー値をそのまま次へ送るのではなく、まず状態 `z` に要約してから行動 `a` を入れて未来を考える、という発想です。世界モデルの大半は、この『いったん裏の状態へまとめてから未来を進める』設計をさまざまな角度から見ています。

## 状態を進めて観測へ戻す

世界モデルの最小形は、状態 `z` を持ち、行動 `a` で次の状態へ進め、必要なときだけ観測 `x` に戻す流れです。ここでは 2 次元の状態を 1 ステップずつ進めて、行動を変えると将来の観測がどう変わるかを見ます。

In [ ]:
import numpy as np

transition = np.array([[0.9, 0.2], [0.0, 0.8]])
action_effect = np.array([0.4, -0.1])
observe = np.array([[1.2, 0.0], [0.2, 0.7]])

def rollout(z0, actions):
    states = [z0]
    observations = [observe @ z0]
    z = z0.copy()
    for a in actions:
        z = transition @ z + action_effect * a
        states.append(z)
        observations.append(observe @ z)
    return np.array(states), np.array(observations)

z0 = np.array([0.2, -0.1])
for actions in ([0, 0, 0], [1, 1, 1], [1, -1, 1]):
    states, observations = rollout(z0, actions)
    print('actions =', actions)
    print('  final state       =', np.round(states[-1], 3))
    print('  final observation =', np.round(observations[-1], 3))

ここで変えているのは行動列だけです。同じ初期状態でも、行動を変えると未来の状態と観測が変わります。世界モデルを学ぶ理由は、このような「もしこの行動を選んだら」を実環境の外で試せるようにするためです。

## どこでつまずきやすいか

- 1 ステップ予測が当たるだけで十分だと思い込む
- 復元品質と計画品質を同じ指標で見ようとする
- シミュレーションの誤差と観測化の誤差を混同する
- SSM と Transformer の比較を性能順位だけで読んでしまう

世界モデルでは『何を予測しているか』『その予測を何に使うか』を分けて読むのが重要です。さらに『そのテーマは世界モデルのどの部品を強くしているのか』まで言えれば、細かい用語に引っ張られにくくなります。

In [ ]:
checks = [
    'stateとobservationを分けているか',
    '1-stepとmulti-stepを分けて評価しているか',
    '行動を入れた比較になっているか',
]
for check in checks:
    print('-', check)

この 3 つは世界モデルの共通チェックリストです。どのテーマでも、この観点へ戻ると『いま何を理解すべきか』が見えやすくなります。初学者が途中で迷ったときも、まずこの 3 つへ戻れば読み筋を立て直しやすくなります。まずこの 3 つに答えられれば、そのテーマの中心は掴めています。

## 誤差の場所を分ける

世界モデルの失敗は、状態の進め方、観測への戻し方、計画スコアの作り方のどこかで起きます。平均誤差だけではなく、どの部品が崩れているかを分けて見ると、直すべき場所がはっきりします。


In [ ]:
parts = {
    'transition': [0.04, 0.06, 0.18, 0.22],
    'decoder': [0.03, 0.05, 0.04, 0.06],
    'reward_model': [0.02, 0.08, 0.15, 0.19],
}
summary = {
    name: sum(values) / len(values)
    for name, values in parts.items()
}
for name, value in sorted(summary.items(), key=lambda x: x[1], reverse=True):
    print(name, round(value, 4))
print('first repair target =', max(summary, key=summary.get))

遷移誤差が大きいなら、観測復元を磨いてもロールアウトは安定しません。報酬モデルが弱いなら、未来状態が当たっていても計画の順位づけを間違えます。部品ごとに誤差を見る習慣が、世界モデルを実装可能な設計へ近づけます。


## 1 ステップ評価と長期評価を分ける

1 ステップ先だけが少し当たっていても、その予測を繰り返すと誤差が積み上がります。世界モデルでは、短期の予測精度と、ロールアウトしても崩れにくいかを分けて見ます。

In [ ]:
true_next = np.array([1.0, 1.2, 1.4, 1.6])
one_step_pred = np.array([1.02, 1.18, 1.43, 1.55])
rollout_pred = np.array([1.02, 1.10, 1.22, 1.30])

one_step_mae = np.mean(np.abs(one_step_pred - true_next))
rollout_mae = np.mean(np.abs(rollout_pred - true_next))
print('one-step MAE =', round(one_step_mae, 4))
print('rollout MAE  =', round(rollout_mae, 4))
print('gap          =', round(rollout_mae - one_step_mae, 4))

この差が大きいと、単発の予測はよく見えても、計画に使うと早く崩れる可能性があります。世界モデルの評価では、見た目の復元だけでなく、予測を何回も使ったときの安定性まで見る必要があります。

## 行動感度を見る

行動を変えても予測がほとんど変わらないモデルは、計画には使いにくくなります。次の例では、同じ初期状態から複数の行動列を流し、最終状態がどれくらい広がるかを測ります。


In [ ]:
action_sets = [
    [0, 0, 0, 0],
    [1, 1, 1, 1],
    [1, -1, 1, -1],
    [-1, -1, 0, 1],
]
final_states = []
for actions in action_sets:
    states, _ = rollout(np.array([0.2, -0.1]), actions)
    final_states.append(states[-1])
final_states = np.array(final_states)
spread = final_states.max(axis=0) - final_states.min(axis=0)
print('final_states =')
print(np.round(final_states, 3))
print('action_sensitivity =', np.round(spread, 3))

行動感度が小さすぎる場合、モデルは環境の変化を予測しているように見えても、行動選択の違いを表現できていない可能性があります。計画に使う世界モデルでは、予測精度だけでなく、行動の違いが未来に反映されるかを確認します。


## 到達目標

到達点は、世界モデルを『生成モデルの派生』としてだけでなく、『予測を使って計画するための分解』として説明できることです。内部状態、状態の進み方、観測復元、ロールアウト、計画評価の役割を言い分けられれば、かなり大きな前進です。さらに『いま読んでいるテーマは、その分解のどの部品を強くしているのか』まで言えれば、世界モデルの芯はかなり掴めています。一度で全部を覚えるより、内部状態、遷移、観測復元、ロールアウト、計画評価のどれを扱っているかを自分の言葉で言い直せる状態を目指します。